# llm-traffic-replay: smoke test (client correctness only)
Self-contained copy of the repo (v0.4.0, 42 files, 187 tests), unpacked to the driver and run against a **pay-per-token** endpoint in this workspace at 1-6 QPS with small prompts.

**What this run proves:** auth path, streaming, TTFT-on-first-content capture, usage parsing, and which cached-token field this serving stack reports.

**What this run must never be quoted for: latency or performance.** Shared pay-per-token capacity says nothing about a dedicated provisioned throughput endpoint. The PT runs follow `docs/PRODUCTION_TESTING.md` stage 2.

In [ ]:
# Cell 1: unpack the embedded repo to the driver
import base64, json, os
from pathlib import Path

PAYLOAD = "..."

root = Path("/tmp/llm_traffic_replay")
for rel, text in json.loads(base64.b64decode(PAYLOAD)).items():
    p = root / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
os.chdir(root)
import sys
sys.path.insert(0, str(root))
print("unpacked to", root, "|", sum(1 for _ in root.rglob('*') if _.is_file()), "files")

In [ ]:
# Cell 2: run the full test suite (187 tests) + instrument validation, right here
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/run_tests_stdlib.py"], capture_output=True, text=True)
print(r.stdout[-1200:]);  assert " 0 failed" in r.stdout, "TEST SUITE NOT GREEN, STOP"
r2 = subprocess.run([sys.executable, "-m", "traffic_replay", "validate", "--quiet", "--workdir", "/tmp/trval"], capture_output=True, text=True)
print(r2.stdout[-900:]); assert "VALIDATE: PASS" in r2.stdout, "INSTRUMENT NOT VALID HERE, STOP" 

In [ ]:
# Cell 3: ambient auth + pick a pay-per-token chat endpoint (no tokens leave this notebook)
import json, urllib.request
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
HOST = "https://" + ctx.browserHostName().get()
TOKEN = ctx.apiToken().get()

req = urllib.request.Request(HOST + "/api/2.0/serving-endpoints", headers={"Authorization": f"Bearer {TOKEN}"})
eps = json.loads(urllib.request.urlopen(req).read()).get("endpoints", [])
chat = [e["name"] for e in eps
        if e.get("name","").startswith("databricks-")
        and e.get("task","") in ("llm/v1/chat","chat/completions","agent/v1/chat")]
print(len(eps), "endpoints;", len(chat), "pay-per-token chat candidates")
print(chat[:12])
# prefer a glm or gpt-oss endpoint when the workspace has one
ENDPOINT = next((n for n in chat if "glm" in n), None) or next((n for n in chat if "gpt-oss" in n), None) or chat[0]
print("selected:", ENDPOINT)

In [ ]:
# Cell 4: 60-second smoke replay at 1-6 QPS, small prompts, capped outputs
import json
from traffic_replay.runner import RunConfig, run

rc = RunConfig(
    profile_path="configs/profile_validation_small.json",
    endpoint={"base_url": HOST,
              "path": f"/serving-endpoints/{ENDPOINT}/invocations",
              "auth_token_env": "UNUSED"},
    duration_s=60, qps_base=2.0, qps_burst=5.0, qps_min=1.0, qps_max=6.0,
    max_concurrency=16, cpt=4.0, calibrate_n=6,
    out_dir="/tmp/tr_smoke", title=f"smoke vs {ENDPOINT} (client correctness only)",
    label="SMOKE TEST on shared pay-per-token capacity: NOT performance evidence.",
    max_output_tokens_cap=24)
out = run(rc, token_override=TOKEN)
print(json.dumps(out["summary"]["ttft_ms"], indent=1))
print("achieved cache:", json.dumps(out["summary"]["achieved_cache_fraction"], indent=1))
print("token targeting:", json.dumps(out["summary"]["token_targeting"], indent=1))

In [ ]:
# Cell 5: the report, verbatim
from pathlib import Path
print(Path(out["out_dir"], "report.md").read_text())